## Data ingestion


In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="i am trying to build rag",
    metadata={
        "source": "rag.txt",
        "author":"udit",
        "page":"14",
        "date_created":"2026-03-02"
    }
)
doc   

Document(metadata={'source': 'rag.txt', 'author': 'udit', 'page': '14', 'date_created': '2026-03-02'}, page_content='i am trying to build rag')

In [3]:
import os
os.makedirs("../data/text_files", exist_ok=True)

In [4]:
sample_texts ={
    "../notebook/data/text_files/my_file.txt": """
Retrieval Augmented Generation (RAG) is a technique used in artificial intelligence to improve the accuracy of large language models by combining them with external knowledge sources. Instead of relying only on the model’s trained knowledge, RAG retrieves relevant information from documents, databases, or files and provides that context to the model before generating a response. Python is commonly used to build RAG systems because of its powerful AI and data libraries such as LangChain, FAISS, and OpenAI. In a typical RAG pipeline, documents are first loaded and split into smaller chunks, then converted into embeddings and stored in a vector database. When a user asks a question, the system searches the vector database to retrieve the most relevant chunks and sends them along with the query to the language model, which generates a more accurate and context-aware answer.
"""
}
for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

In [5]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../notebook/data/text_files/my_file.txt",encoding="utf-8")
document=loader.load()
document

d:\raglearn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../notebook/data/text_files/my_file.txt'}, page_content='\nRetrieval Augmented Generation (RAG) is a technique used in artificial intelligence to improve the accuracy of large language models by combining them with external knowledge sources. Instead of relying only on the model’s trained knowledge, RAG retrieves relevant information from documents, databases, or files and provides that context to the model before generating a response. Python is commonly used to build RAG systems because of its powerful AI and data libraries such as LangChain, FAISS, and OpenAI. In a typical RAG pipeline, documents are first loaded and split into smaller chunks, then converted into embeddings and stored in a vector database. When a user asks a question, the system searches the vector database to retrieve the most relevant chunks and sends them along with the query to the language model, which generates a more accurate and context-aware answer.\n')]

In [6]:
from langchain_community.document_loaders import DirectoryLoader
dir_loader = DirectoryLoader(
"../notebook/data/text_files",
glob="**/*.txt",
loader_cls=TextLoader,
loader_kwargs={'encoding':'utf-8'},
show_progress=False
 )
load_doc = dir_loader.load()
load_doc

[Document(metadata={'source': '..\\notebook\\data\\text_files\\my_file.txt'}, page_content='\nRetrieval Augmented Generation (RAG) is a technique used in artificial intelligence to improve the accuracy of large language models by combining them with external knowledge sources. Instead of relying only on the model’s trained knowledge, RAG retrieves relevant information from documents, databases, or files and provides that context to the model before generating a response. Python is commonly used to build RAG systems because of its powerful AI and data libraries such as LangChain, FAISS, and OpenAI. In a typical RAG pipeline, documents are first loaded and split into smaller chunks, then converted into embeddings and stored in a vector database. When a user asks a question, the system searches the vector database to retrieve the most relevant chunks and sends them along with the query to the language model, which generates a more accurate and context-aware answer.\n')]

In [7]:
import os 
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
def load_pdf(pdf_directory):
    """load all pdf files form directory"""
    all_documents =[]
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files")

    for pdf_file in pdf_files:
        print(f"Processing -{pdf_file.name}")

        
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"error processing : {e}")

    print(f"\ntotal documents loaded: {len(all_documents)}")
    return all_documents
        
all_pdf_documents = load_pdf("data/text_files")

Found 1 PDF files
Processing -Elite_Backend_DSA_Roadmap_2025.pdf
loaded 8 pages

total documents loaded: 8


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2025-10-02T06:56:44+00:00', 'source': 'data\\text_files\\Elite_Backend_DSA_Roadmap_2025.pdf', 'file_path': 'data\\text_files\\Elite_Backend_DSA_Roadmap_2025.pdf', 'total_pages': 8, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-02T06:56:44+00:00', 'trapped': '', 'modDate': "D:20251002065644+00'00'", 'creationDate': "D:20251002065644+00'00'", 'page': 0, 'source_file': 'Elite_Backend_DSA_Roadmap_2025.pdf', 'file_type': 'pdf'}, page_content='Week 1: Master Node.js Beyond Basics\nWeek 2: Database Mastery\nWeek 3: API Design & Security\nElite Backend Engineer Roadmap & DSA Mastery\nPlan\nFrom Node.js Foundation to Stanford/MIT Level Backend Engineering\nTable of Contents\n1. Backend Engineering Elite Roadmap\n2. Data Structures & Algorithms (DSA) Complete Guide\n3. Competitive Programming Strategy (LeetCode, CodeChef, Codeforces)\n4. System Design & Ar

In [10]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split document into small parts"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )

    split_documents = text_splitter.split_documents(documents)
    print(f"Split{len(documents)} docs into {len(split_documents)}chunks")
    
    if split_documents:
        print(f"\nexample chunk:"),
        print(f"content: {split_documents[0].page_content[:200]}...")
        print(f"metadata: {split_documents[0].metadata}")

    
    return split_documents

In [11]:
chunks = split_documents(all_pdf_documents)
chunks

Split8 docs into 15chunks

example chunk:
content: Week 1: Master Node.js Beyond Basics
Week 2: Database Mastery
Week 3: API Design & Security
Elite Backend Engineer Roadmap & DSA Mastery
Plan
From Node.js Foundation to Stanford/MIT Level Backend Engi...
metadata: {'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2025-10-02T06:56:44+00:00', 'source': 'data\\text_files\\Elite_Backend_DSA_Roadmap_2025.pdf', 'file_path': 'data\\text_files\\Elite_Backend_DSA_Roadmap_2025.pdf', 'total_pages': 8, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-02T06:56:44+00:00', 'trapped': '', 'modDate': "D:20251002065644+00'00'", 'creationDate': "D:20251002065644+00'00'", 'page': 0, 'source_file': 'Elite_Backend_DSA_Roadmap_2025.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2025-10-02T06:56:44+00:00', 'source': 'data\\text_files\\Elite_Backend_DSA_Roadmap_2025.pdf', 'file_path': 'data\\text_files\\Elite_Backend_DSA_Roadmap_2025.pdf', 'total_pages': 8, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-10-02T06:56:44+00:00', 'trapped': '', 'modDate': "D:20251002065644+00'00'", 'creationDate': "D:20251002065644+00'00'", 'page': 0, 'source_file': 'Elite_Backend_DSA_Roadmap_2025.pdf', 'file_type': 'pdf'}, page_content='Week 1: Master Node.js Beyond Basics\nWeek 2: Database Mastery\nWeek 3: API Design & Security\nElite Backend Engineer Roadmap & DSA Mastery\nPlan\nFrom Node.js Foundation to Stanford/MIT Level Backend Engineering\nTable of Contents\n1. Backend Engineering Elite Roadmap\n2. Data Structures & Algorithms (DSA) Complete Guide\n3. Competitive Programming Strategy (LeetCode, CodeChef, Codeforces)\n4. System Design & Ar

# EMbedding and vector db

In [12]:
import numpy as np
import uuid
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from typing import List , Dict , Tuple , Any
from sklearn.metrics.pairwise import cosine_similarity


In [14]:
class EmbeddingManager:
    """ handles doc strings"""
    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model=None
        self._load_model()
        
    def _load_model(self):
        """Load sentence transformers"""   
        try:
            print(f"loading the embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded successfully : embedding dimension : {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model : {self.model_name}:{e}")
            raise


    def generated_embedding(self,texts:List[str]) -> np.ndarray:
        """
        Generate embeddings for lists of texts
        
        
        """
        print(f"genrating embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"genrate embedding with shape :{embeddings.shape}")
        return embeddings

embeddin_manager=EmbeddingManager()
embeddin_manager

loading the embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3549.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded successfully : embedding dimension : 384


# VectorStore

In [ ]:
class vectordatabase:

    def __init__(self,collection_name:str ="pdf_documents",persist_directory:str ="/data/vector_store"):
        
        self.collection_name=collection_name
        self.persist_directory =persist_directory
        self.client = None
        self.collection = None
        self._intialize_store()

    def _initialize_store(self):


